In [1]:
!pip install tensorflow

In [2]:
!pip install pillow numpy

In [3]:
!pip install fastapi uvicorn

In [13]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np

# Load raw datasets
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    "dataset",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    "dataset",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

# Save class names before mapping
class_names = train_ds_raw.class_names
num_classes = len(class_names)
np.save("class_names.npy", class_names)
print("Detected classes:", class_names)

# Data augmentation
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(factor=0.2),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds_raw.map(lambda x, y: (data_augmentation(x, training=True), y)).prefetch(AUTOTUNE)
val_ds = val_ds_raw.prefetch(AUTOTUNE)

# Base model
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

# Build model (functional API for clarity)
inputs = layers.Input(shape=(224, 224, 3))
x = tf.keras.applications.efficientnet.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
checkpoint = ModelCheckpoint("best_plant_diagnosis.keras", save_best_only=True)

# Train
print("Training robust Plant Diagnosis Model...")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stop, reduce_lr, checkpoint]
)

print("Training complete. Best model saved as best_plant_diagnosis.keras")

# Save final model
model.save("plant_diagnosis.keras")


Found 79129 files belonging to 18 classes.
Using 63304 files for training.
Found 79129 files belonging to 18 classes.
Using 15825 files for validation.
Detected classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy', 'fruit_apple_fresh', 'fruit_apple_rotten', 'fruit_tomato_fresh', 'fruit_tomato_rotten']
Training robust Plant Diagnosis Model...
Epoch 1/30
1979/1979 ━━━━━━━━━━━━━━━━━━━━ 5287s 3s/step - accuracy: 0.7738 - loss: 0.7185 - val_accuracy: 0.8726 - val_loss: 0.3893 - learning_rate: 5.0000e-04
Epoch 2/30
1979/1979 ━━━━━━━━━━━━━━━━━━━━ 5523s 3s/step - accuracy: 0.8501 - loss: 0.4535 - val_accuracy: 0.8931 - val_loss: 0.3204 - learning_rate: 5.000

In [17]:
import numpy as np
np.save("class_names.npy", class_names)
print("Class names saved:", class_names)


Class names saved: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy', 'fruit_apple_fresh', 'fruit_apple_rotten', 'fruit_tomato_fresh', 'fruit_tomato_rotten']


In [2]:
import requests

# Gateway URL (Express forwards to FastAPI)
GATEWAY_URL = "http://192.168.1.152:8000/predict"   # use LAN IP if testing from another device

# Path to a test image
image_path = "test_dataset/fresh/R.jpg"   # replace with your actual image path

def test_prediction():
    try:
        with open(image_path, "rb") as f:
            files = {"file": f}
            response = requests.post(GATEWAY_URL, files=files)

        print("Status Code:", response.status_code)
        print("Response JSON:", response.json())
    except Exception as e:
        print("Error during request:", str(e))

if __name__ == "__main__":
    test_prediction()


Status Code: 200
Response JSON: {'success': True, 'message': 'This is a Apple leaf. Disease: Apple scab.', 'status': 'Apple Leaf – Scab', 'confidence': '36.02%', 'danger': 'Leaf diseases reduce crop yield but are not directly harmful if leaves are not eaten.', 'advice': 'Use resistant varieties and apply fungicide sprays.', 'warning': 'Low confidence (36.02%). Please verify with a clearer image.'}
